# Annual Report Extractor — Historical Extraction

Upload several statement PDFs at once (Balance Sheet / Income Statement / Cash Flow, across multiple years) and get back **one Excel workbook with a multi-year historical block** — line items down the side, fiscal years across the top — ready to paste into a valuation model.

Run each cell in order by clicking the ▶ button on its left.

**Two things this does that matter for accuracy:**
- It reads the column headers to work out which column is which — so on a quarterly results filing it takes the *annual consolidated* column, not a quarter.
- It runs footing checks (Revenue + Other Income = Total Income, Assets = Equity + Liabilities, etc.) and reports any that don't tie out.

## 1. Install system dependencies

Tesseract reads scanned pages; Poppler converts PDF pages into images for it.

In [ ]:
!apt-get -qq update && apt-get -qq install -y tesseract-ocr poppler-utils
print("done")

## 2. Upload the project zip

Select `annual-report-extractor.zip` when the file picker appears.

*(Run this cell only once per session. Running it twice unzips a copy inside a copy.)*

In [ ]:
import os
from google.colab import files

os.chdir("/content")
uploaded = files.upload()  # select annual-report-extractor.zip
zip_name = next(iter(uploaded))

!rm -rf /content/annual-report-extractor
!unzip -q -o "{zip_name}" -d /content
os.chdir("/content/annual-report-extractor")
print("now in:", os.getcwd())

## 3. Install the package

In [ ]:
!pip install -q .
print("installed")

## 4. Upload your statement PDFs

**Select all of them at once** (hold Ctrl / Cmd to multi-select).

Keep the filenames descriptive — names like `2024-25 Balance Sheet.pdf` or `2022-23 Cash Flow.pdf` let the tool work out both the statement type and the fiscal year without guessing.

In [ ]:
import shutil

INPUT_DIR = "/content/statements"
shutil.rmtree(INPUT_DIR, ignore_errors=True)
os.makedirs(INPUT_DIR, exist_ok=True)

uploaded_reports = files.upload()  # select ALL your statement PDFs together
for name in uploaded_reports:
    shutil.move(name, os.path.join(INPUT_DIR, name))

print(f"{len(uploaded_reports)} file(s) ready:")
for name in sorted(uploaded_reports):
    print(" -", name)

## 5. Run the extraction

Scanned pages need OCR, which takes roughly 20–40 seconds per page — so a batch of ten files can take several minutes. The progress prints as it goes.

Set `BASIS` to `"standalone"` if you want standalone rather than consolidated figures.

In [ ]:
import glob
from arx.batch import process_file, build_historical
from arx.validate import run_checks, summarize

BASIS = "consolidated"   # or "standalone"

results = []
for path in sorted(glob.glob(os.path.join(INPUT_DIR, "*"))):
    print(f"Reading {os.path.basename(path)} ...")
    try:
        found = process_file(path)
    except Exception as exc:
        print(f"   ! failed: {exc}")
        continue
    if not found:
        print("   ! no statement recognised in this file")
    for r in found:
        dated = "dated" if r.layout.confident else "UNDATED"
        print(f"   {r.statement}: {len(r.standardized)} line items, "
              f"{len(r.layout.columns)} columns ({dated})")
    results.extend(found)

historical = build_historical(results, basis_preference=BASIS)
checks = run_checks(historical.tables)
print("\nConsistency checks:", summarize(checks))

## 6. Look at the historical tables

In [ ]:
from IPython.display import display

for statement, table in historical.tables.items():
    print(f"\n=== {statement} ===")
    display(table)

## 7. Check what didn't tie out

Anything marked **FAIL** below means an extracted figure doesn't foot — verify that one against the source page before using it. Anything in **Needs Review** was deliberately left out of the historicals because its column position was uncertain.

In [ ]:
failed = checks[checks["status"] == "FAIL"]
print(f"Failed checks: {len(failed)}")
if len(failed):
    display(failed)

if not historical.conflicts.empty:
    print("\nFiles disagreeing on the same year:")
    display(historical.conflicts)

if not historical.needs_review.empty:
    print("\nExcluded from historicals (column position uncertain):")
    display(historical.needs_review)

## 8. Download the workbook

In [ ]:
from arx.export.excel_writer import write_historical_workbook

OUTPUT = "/content/historical.xlsx"
write_historical_workbook(OUTPUT, historical, checks)
files.download(OUTPUT)